# Kinnow ERP assistant: Kaggle control notebook

**What this notebook is for.** Kinnow is an assistant that answers questions about GST, payroll and invoice
matching inside an ERP system. We start from a general-purpose AI model (Qwen2.5-7B) and train it further on
questions and answers that Tejas and our team reviewed. This notebook is where we start those training and
scoring jobs on Kaggle's free graphics cards (GPUs).

**What it does not do.** It does not change the live product. Nothing in it runs by itself: you run the cells you
need, one at a time.

**Where the real code lives.** In our GitHub project `BonsAI-Agents/Kinnow_LLM`. The code cells below only start
the project's scripts, so this notebook stays short and the real logic stays under version control.

**Our goal** (set in `kinnow-production-plan-v1.md`): the model should pass **78%** of our test questions overall
(GST 65%, Payroll 82%, Invoice 86%) and wrongly refuse a real question less than 3% of the time.

## How to read it

| Section | What you will find |
|---|---|
| 1 | What to set up once on Kaggle, and the five helper tools |
| 2 | Every model version in one table, with scores |
| 3 to 7 | One short story per version: what changed, what happened, what we learned |
| 8 | The buttons: train a version, score a version, check progress, read results |
| 9 | Problems we have hit and how to fix them |
| 10 | Where each file is in the project |
| 11 | Plain-word glossary |

## 1. One-time setup

Do these once each time you open the notebook on Kaggle.

1. **Turn on two graphics cards.** In the session options choose **GPU T4 x2**. Training and scoring need them.
2. **Turn on the internet.** The jobs download code and models.
3. **Add the Hugging Face key.** Open **Add-ons > Secrets** and attach a secret named `HF_TOKEN` (a Hugging Face key that is allowed to write).
   Hugging Face is the website where we store trained models. The key is never shown or saved to a file.
4. **Know the limits.**
   - A Kaggle session ends after about 40 minutes with no activity, or after 12 hours.
     **When it ends, everything in `/kaggle/working` is deleted** (downloaded code, the Python setup, result files).
     Trained models are safe because training uploads them to Hugging Face. Result files are not safe, so read or
     download them before the session ends.
   - Kaggle gives about **30 GPU hours a week**. Training one version uses about 4 hours; scoring one version about 45 minutes.
5. **Trained models are stored privately.** New models go to **private** Hugging Face repos, so an unchecked model cannot be published by accident.

### The five helper tools (run the next cell first, every session)

The cell below does not train or score anything. It only defines five small tools that the later cells use.

| Tool | What it does | When to use it |
|---|---|---|
| `start_job(script, ...)` | Downloads one of our scripts from GitHub and runs it **in the background**, so you can close the tab. Writes everything it prints into a log file | To start training or scoring (section 8) |
| `job_status(log)` | Shows which milestones the job has reached (`TRAIN DONE`, `EVAL SAVED`, `JOB DONE`) and the last line of its log. Warns if an error appeared | To check on a running job |
| `show_log(log, lines)` | Prints the last lines of a job's log, without the long technical warnings | When `job_status` shows an error, or you want details |
| `show_results(log)` | Prints the score table the scoring job produced | When a scoring job is finished |
| `show_failures(results_file)` | Lists the test questions the model got wrong and which required words its answer lacked | To study what to fix |

In [ ]:
# The five helper tools. Run this cell first in every session. It only defines tools; it does not start anything.
import os
import subprocess
import json

# Where our scripts live on GitHub, and Kaggle's writable folder (everything here is lost when the session ends)
RAW = "https://raw.githubusercontent.com/BonsAI-Agents/Kinnow_LLM/main/scripts/"
W = "/kaggle/working"


def start_job(script, log="job.log", **env_vars):
    """Download scripts/<script> from GitHub and run it in the background. Returns the running process.

    The Hugging Face key is read from Kaggle Secrets and handed to the job only; it is never printed.
    Extra keyword arguments become settings for the job, for example VERSION="v5"."""
    from kaggle_secrets import UserSecretsClient
    subprocess.run(f"curl -s {RAW}{script} -o {W}/{script}", shell=True, check=True)      # fetch the newest script
    env = dict(os.environ, HF_TOKEN=UserSecretsClient().get_secret("HF_TOKEN"), **env_vars)
    p = subprocess.Popen(f"python {W}/{script} > {W}/{log} 2>&1", shell=True, env=env, start_new_session=True)
    print(f"started {script} (pid {p.pid}); log: {W}/{log}")
    return p


def job_status(log="job.log"):
    """Which milestones the job has reached, plus the last line of its log."""
    path = f"{W}/{log}"
    if not os.path.exists(path):
        print("no log yet")
        return
    text = open(path, errors="replace").read()
    for mark in ("TRAIN DONE", "EVAL SAVED", "JOB DONE"):
        print(("done    " if mark in text else "pending ") + mark)
    if "Traceback" in text:
        print("ERROR: the log contains an error report; run show_log() to read it")
    last = text.replace("\r", "\n").strip().splitlines()[-1][:150] if text.strip() else ""
    print("last line:", last)


def show_log(log="job.log", lines=40):
    """The last lines of a job's log, without the long library warnings."""
    text = open(f"{W}/{log}", errors="replace").read().replace("\r", "\n").splitlines()
    text = [t for t in text if "warn" not in t.lower()]
    print("\n".join(t[:160] for t in text[-lines:]))


def show_results(log="job.log"):
    """The score table printed by a finished scoring job (overall, per topic, Hindi, multi-turn, speed)."""
    text = open(f"{W}/{log}", errors="replace").read().splitlines()
    for i, line in enumerate(text):
        if "Full eval (" in line:
            print("\n".join(text[i:i + 17]))
            return
    print("no score table in the log yet")


def show_failures(results_json, limit=200):
    """One line per failed question: topic, question label, and the required words the answer was missing.
    results_json is a file such as /kaggle/working/v5_full.json written by a scoring job."""
    norm = lambda s: s.lower().replace(",", "")
    data = json.load(open(results_json))
    failed = [g for g in data["golden"] if not g["passed"] and g["category"] != "REFUSAL"]
    print(f"{len(failed)} failed questions")
    for g in failed[:limit]:
        missing = [w for w in g["must_contain"] if norm(w) not in norm(g["response"])]
        wrong = [w for w in g.get("must_not_contain", []) if norm(w) in norm(g["response"])]
        print(f"{g['category'][:3]} {g['tag'][:40]:<40} missing={missing} wrong={wrong}")

## 2. All versions in one table

Each row is one trained model. "Score" is the share of our **192 English test questions** the model passed
(`eval/golden_set.jsonl`). A question passes when the answer contains the words we require and none we forbid.
"With lookup" means the model was shown the most relevant approved notes before answering (explained in section 4).

| Version | What changed | Where it is stored (private, Hugging Face) | Score |
|---|---|---|---|
| Untrained | The original Qwen2.5-7B model, no extra training | none | 22.9% |
| v1 | First training run. The answers never stopped (a setting bug) | not recorded | not recorded |
| v2 | Bug fixed. Trained on Tejas's 500 reviewed rows plus our own rows that do not conflict | `KaraPatelDev/kinnow-v2-qwen25-sft-adapter` | 41.7%; 52.1% with lookup |
| v3 | Added the lookup notes and 312 rows with calculations done by computer | `KaraPatelDev/kinnow-v3-qwen25-sft-adapter` | 54.7% with lookup |
| v4 | Tejas's corrected 500 rows plus 89 new Payroll and Invoice rows | `KaraPatelDev/kinnow-v4-qwen25-sft-adapter` | 50.0% with lookup |
| v4 + new answer features | Same v4 model, plus a better lookup, calculators and a "Sources" line. No new training | same as v4 | **57.8%** (52.6% without the Sources line) |

**How to read the scores.** Each topic has only about 50 questions, so one question is worth about 2 points.
Running the same test twice can move a few answers: v3 scored 55.6% on the Hindi questions one time and 63.0% on an
identical re-run, with the same overall score. Treat differences of a few points as chance unless they repeat.

**Where we stand against the goal:** 57.8% against 78%, which is 20 points short. If a human accepted the answers
that fail only because an exact word is missing (for example "RCM" instead of "reverse charge"), the real figure
would be roughly 69%. That is a rough automatic estimate. Tejas's review (section 7) will settle it.

### Full results

Overall / GST / Invoice / Payroll, in percent.

| Run | Overall | GST | Invoice | Payroll | Notes |
|---|---|---|---|---|---|
| Untrained, no lookup | 22.9 | 1.9 | 5.8 | 14.0 | Starting point |
| v2, no lookup | 41.7 | 29.6 | 26.9 | 30.0 | |
| v2 with lookup (4 notes) | 52.1 | 42.6 | 40.4 | 42.0 | The lookup adds about 10 points |
| v3 with lookup, original questions | 54.7 | 48.1 | 44.2 | 42.0 | Wrong refusals 0%; off-topic refusals 97.2% |
| v3 with lookup (6 notes) | 53.6 | 40.7 | 44.2 | 46.0 | More notes did not help, so we keep 4 |
| v3, updated questions and notes | 52.6 | 46.3 | 36.5 | 44.0 | Same questions as v4 |
| v4 with lookup (4 notes) | 50.0 | 38.9 | 38.5 | 42.0 | Within chance of v3 |
| v4, better lookup and calculators | 52.6 | 38.9 | 40.4 | 48.0 | Without the Sources line |
| v4, plus the Sources line | **57.8** | 44.4 | 50.0 | 52.0 | Wrong refusals 0%; off-topic refusals 97.2% |

**Fact check.** We also check 50 facts from Tejas's list. With the lookup, v3 and v4 both get 48 of 50 right. Without
it, v3 gets 38 and v4 gets 34.

## 3. v1: the first run (history)

**What it was.** The first time we trained the model on our own reviewed questions and answers
(`data/v1_training`). We used a light training method (called QLoRA) that adjusts a small part of the model, so it
fits on two small graphics cards.

**What went wrong.** A training setting (the "padding" marker) was the same as the marker that means "the answer
ends here". The training then skipped that marker, so the model never learned to stop. Its answers ran on and
rambled, and the scores meant nothing.

**The fix** (made in v2): a different padding marker, plus a safety check in `training/sft_train.py` that stops the
run if the end-of-answer marker is not being learned.

**Kept for reference.** `data/v1_training/raw.jsonl` is still used by our automated tests to decide which topic a
question belongs to, so do not edit it. No v1 scores were recorded.

## 4. v2: the bug fix and Tejas's 500 rows

**Training data.** `scripts/merge_tejas_data.py` combined Tejas's 500 reviewed rows with our earlier rows that do not
conflict. Its rules: where the same question appears twice, Tejas's version wins; all "please refuse this" examples
are kept; and earlier rows built on the old income-tax law (Income-tax Act 1961, Forms 16 and 24Q, old TDS section
numbers) are set aside in `dropped_old_regime.jsonl`, because the new Act (Sections 392 and 393, Forms 130 and 138)
replaced them.

**Training run.** Same settings as v1 with the bug fixed. It ran at about 25 seconds per step on two T4 cards. The
first progress check crashed with an out-of-memory error, because it tried to check 8 long answers at once; it now
checks one at a time.

**Scores.**

| Run | Overall | GST | Invoice | Payroll |
|---|---|---|---|---|
| Untrained model | 22.9 | 1.9 | 5.8 | 14.0 |
| v2 | 41.7 | 29.6 | 26.9 | 30.0 |
| v2 with lookup | 52.1 | 42.6 | 40.4 | 42.0 |

**What we learned.**
- Training added about 19 points over the untrained model. The lookup added about 10 more.
- Answers became short (about 41 words), stopped properly, and took about 6 seconds each.
- Most remaining failures were missing exact section numbers and figures, not a problem with the scoring.

**The lookup, in plain words.** Before the model answers, the system searches our approved notes for the 4 that
best match the question and shows them to the model. The code is `guardrails/retrieval.py`. The notes are built by
`scripts/build_kb.py` from the training data, leaving out refusal examples, computer-generated rows, and any
question that is also in the test set (so the test stays fair).

## 5. v3: lookup notes and computer-made calculation rows

**New training rows.**
- 312 rows whose numbers are worked out by a program (`scripts/generate_calc_rows.py`): gratuity, provident fund,
  health insurance contribution (ESI), income tax under the new regime, leave encashment, interest, credit reversal
  rules, e-invoice limits, TDS under GST, and composition rates. Because a program calculates every figure, the
  arithmetic in these rows cannot be wrong. The rules themselves come from Tejas's list.
- Rows on interest for wrongly claimed tax credit (Section 50(3)) at **24%**. These were replaced by 18% in v4.

**New check.** `eval/fact_checks.jsonl` and `eval/run_fact_check.py`: 50 facts from Tejas's list, each tested
automatically.

**Results** (with a lookup of 4 notes, original questions): overall 54.7, GST 48.1, Invoice 44.2, Payroll 42.0.
Wrong refusals 0%; off-topic refusals 97.2%. Fact check: 48 of 50 with the lookup, 38 without.
Payroll did not improve even after adding 150 Payroll rows. Our Payroll test questions are mostly tricky cases with
several conditions, while those rows covered simple formulas.

**Lookup size test.** Showing 6 notes scored 53.6, lower than 4 notes (54.7), so the lookup stays at 4.

## 6. v4: Tejas's corrected data and the Section 50(3) decision

**Training data** (2,691 rows in total, combined by `scripts/merge_tejas_data.py`).
- `data/tejas/kinnow_sft_500_v2.jsonl`: Tejas's 500 rows with Section 50(3) interest corrected to 18%. The
  meal-voucher row was replaced by a question on overtime and provident fund.
- `data/tejas/kinnow_sft_payroll_invoice_v2.jsonl`: 89 new worked examples (44 Payroll, 45 Invoice). They include
  traps such as amounts Rs 1 over a limit, employees who joined mid-year, late credit notes, and tax-rate changes. Each
  answer names the exact section, rule or form.
- Our own earlier rows that said 24% for Section 50(3) are held out (`held_for_verification.jsonl`).
- The calculation generator now writes 18%, and four test questions expect 18%.

**The 18% decision.** Tejas says interest under Section 50(3) is 18% (citing Notification 14/2022-CT) and that the
Act's 24% is only a ceiling. The owner chose 18% on 21 Nov 2026 and, per the owner, the CA approved it on 22 Nov 2026.
This is logged in `data/tejas_review/DECISIONS.md`. No written note from the CA is stored in the project yet, so add
it there when you have it. A separate web check found secondary sources still saying 24% and read that notification as
the one that introduced Rule 88B (how the interest is calculated); the official text was not read. If the notification
or the CA's note ever says otherwise, change the generator, the four test questions and the fact check, then retrain.

**Training run.** 600 steps, about 3 hours 50 minutes including the progress checks. The error measure on held-back
examples fell at every check: 1.40, 1.23, 1.10, 1.05, 1.02, 1.01 (lower is better).

**Results** (lookup of 4 notes, updated questions): overall 50.0, GST 38.9, Invoice 38.5, Payroll 42.0; off-topic
refusals 94.4%; wrong refusals 0%. Fact check: 48 of 50 with the lookup, 34 without.

**v3 against v4 on identical questions and notes:** v3 scored 52.6 and v4 scored 50.0. That is about 5 questions out
of 192, which is within normal chance. Adding Tejas's 589 rows did not measurably change the score.

## 7. v4 plus three answer features (no new training)

We changed how answers are produced, without retraining. Each feature has an on/off switch in the scoring job
(`--hybrid`, `--calc`, `--cite`) and a setting in the live server.

| Feature | What it does, in plain words | Code |
|---|---|---|
| Better lookup | Finds notes by words **and** by meaning, then merges the two lists. The right notes appear in the top 4 for 70% of test questions, against 64% before | `guardrails/retrieval.py` |
| Calculators | For plain questions (gratuity, provident fund, ESI, income tax, interest) the system works out the number itself and gives it to the model, which then only explains it. It stays silent when a question has more than one amount or unusual wording. It applies to only 5 of the 146 test questions | `guardrails/calculators.py` |
| Sources line | After the answer, adds a line such as "Sources: Section 16(2)(aa), Rule 37", using only references found in the notes shown to the model | `guardrails/retrieval.py` |

**Result on the v4 model** (same questions):

| Setup | Overall | GST | Invoice | Payroll |
|---|---|---|---|---|
| v4, original lookup | 50.0 | 38.9 | 38.5 | 42.0 |
| plus better lookup and calculators | 52.6 | 38.9 | 40.4 | 48.0 |
| plus Sources line | **57.8** | 44.4 | 50.0 | 52.0 |

**How to read this.**
- The better lookup and calculators gave about +2.6 points. The Sources line gave about +5.2.
- The Sources line adds accurate references but does not make the model reason better, and part of its gain is the
  scorer liking exact wording. We chose the lookup variant after looking at the test's required words, so the numbers are
  probably a little optimistic.
- The fair measure of real accuracy is a human review (section below).

**Now in the live server.** `guardrails/answer_support.py` connects all three features to the server. They are on by
default, each can be switched off with a setting, and each falls back to the old behaviour if anything fails. They add
about 0.1 second per question on a laptop. Build the notes file once with
`python scripts/merge_tejas_data.py && python scripts/build_kb.py --embed`, then place `data/knowledge/` next to
`data/centroids.npz`. One known weakness: the Sources line sometimes cites a loosely related note (for example Section
43B(h) on a question about Rule 37), because it takes references from the two best notes.

**Human review (next step).** `data/review/review_sheet_for_tejas.csv` has 50 answers in random order: 40 the scorer
failed and 10 that passed only because of the Sources line. The scorer's verdicts are hidden from the reviewer (they
are in `review_KEY_do_not_share.csv`). Tejas marks each CORRECT, PARTLY or WRONG. Comparing his marks with the
scorer's tells us the real accuracy.

**What the failed answers look like** (automatic sort of 75 failures, `data/review/failure_analysis.md`): 22 have the
right figures but miss an exact word, 26 have different figures, 25 are unclear, and 2 state a forbidden fact.

## 8. The buttons: run a job

Run the helper cell in section 1 first. Every job is one script from the project that runs in the background and
writes a log file in `/kaggle/working`. The script prints milestone markers (`TRAIN DONE`, `EVAL SAVED`, `JOB DONE`)
that `job_status()` reads.

| I want to | Cell | How long |
|---|---|---|
| Train a new version and score it | 8a | about 5 hours |
| Score a version that already exists | 8b | about 15 minutes of setup, then 45 minutes |
| See how a running job is doing | 8c | instant |
| Read the scores of a finished job | 8d | instant |
| See which questions failed and why | 8e | instant |

---

### 8a. Train a new version

**What this cell does.** It starts `scripts/kaggle_train.py`, which then does these steps by itself:
1. Downloads the project and builds a clean Python setup (about 10 minutes on a fresh session).
2. Rebuilds the training data and the lookup notes from the project's files.
3. Creates a **private** Hugging Face repo for the new model.
4. Trains the model for 600 steps (about 3 hours 50 minutes), uploads the result to Hugging Face, and keeps a copy in `/kaggle/working`.
5. Scores the new model on the 192 test questions (about 45 minutes) and saves the answers in `/kaggle/working/<VERSION>_full.json`.

**What you change.** Only `VERSION`, a new label such as `"v5"`. Optional settings: `SEED` (a number; training twice with
different numbers shows how much scores move by chance), `MAX_STEPS`, `FLAGS` (which answer features to score with) and `BASE` (the starting model).

**What you will see.** One line saying it started. Watch it with cell 8c, using `log="train.log"`.

**Cost.** About 5 hours of your weekly GPU time. Do not start it casually.

In [ ]:
# 8a. Train a new version, then score it. Change VERSION to a new label first.
train_job = start_job("kaggle_train.py", log="train.log", VERSION="v5")

### 8b. Score a version that already exists

**What this cell does.** It starts `scripts/kaggle_eval.py`, which sets up the project, rebuilds the lookup notes,
loads the chosen trained model from Hugging Face and answers all 192 test questions. It saves the answers and scores
in `/kaggle/working/<TAG>_full.json`. Use it to re-score a version or to try the answer features without any new training.

**What you change.**
- `ADAPTER`: which trained model to load (the Hugging Face name from the table in section 2).
- `TAG`: a label for this run; it names the result file.
- `FLAGS`: which answer features to use. `--rag 4` shows 4 notes before each answer; `--hybrid` finds notes by words and
  meaning; `--calc` turns on the calculators; `--cite` adds the Sources line (the job then also prints the score
  without it, so you can see its effect). Leave a flag out to switch that feature off.

**What you will see.** One line saying it started. Watch it with cell 8c.

In [ ]:
# 8b. Score an existing version. Change ADAPTER, TAG and FLAGS as needed.
eval_job = start_job(
    "kaggle_eval.py", log="job.log",
    ADAPTER="KaraPatelDev/kinnow-v4-qwen25-sft-adapter",     # which trained model to load
    TAG="v4-check",                                           # label for this run and its result file
    FLAGS="--rag 4 --hybrid --calc --cite",                   # which answer features to use
)

### 8c. Check how a job is doing

**What this cell does.** It reads the job's log file and prints three milestone lines, each either `done` or `pending`,
then the last line the job wrote. If the log contains an error report, it says so; then run `show_log()` to read it.

**How to read it.**
- `TRAIN DONE`: training finished (only for training jobs).
- `EVAL SAVED`: the test answers and scores were saved.
- `JOB DONE`: the whole job finished.
- While training, the last line is a progress bar such as `300/600`; a checkpoint every 100 steps prints an error
  measure (lower is better).
- While scoring, the log stays quiet until the end. That is normal: it takes about 45 minutes.

Use `job_status("train.log")` for a training job and `job_status("job.log")` for a scoring job.

In [ ]:
# 8c. How is the job doing?  (use "train.log" for a training job, "job.log" for a scoring job)
job_status("job.log")

### 8d. Read the scores

**What this cell does.** It finds the score table in a finished job's log and prints it: overall score, one score per
topic, off-topic refusals, wrong refusals, Hindi, multi-turn conversations, and speed. If the job also tested the
Sources line, it prints a line "WITHOUT the Sources line" so you can compare.

In [ ]:
# 8d. Scores from a finished scoring job.
show_results("job.log")

### 8e. See which questions failed and why

**What this cell does.** It opens a result file and prints one line for every question the model failed, with the
words the answer was missing (`missing=[...]`) and any forbidden words it used (`wrong=[...]`).

**How to use it.** Use it to tell two kinds of failure apart: an answer that is right but lacks the exact phrase, and an
answer that is truly wrong. Read a few full answers before trusting a pattern. Change the file name to the run you want.

In [ ]:
# 8e. The questions a finished run got wrong. Change the file name to the run you want.
show_failures("/kaggle/working/v4-check_full.json")

## 9. Problems we have hit, and what to do

| What you see | Why it happens | What to do |
|---|---|---|
| Answers never stop and scores are near zero | A training setting told the model that the "end of answer" marker did not matter | Already fixed in `training/sft_train.py`, with a check that stops the run if it comes back. Do not remove the check |
| The first progress check crashes with "out of memory" | It tried to check 8 long answers at once | It now checks one at a time |
| Training warns about `bf16` | The T4 card does not support that number format | The trainer uses the format T4 supports (fp16) |
| Installing the training tools fails | Kaggle's built-in Python (3.13) is too new for our pinned tool versions | The jobs build their own Python 3.11 setup in `/kaggle/working/venv` |
| A script fails on Kaggle but works on a laptop | Some code only works on Python 3.12 or newer | Write code that also works on 3.11 (for example, single quotes inside f-strings) |
| Everything in `/kaggle/working` is gone | The session ended after about 40 minutes without activity, or after 12 hours | Run section 1 again, then the job. Trained models are safe on Hugging Face. Download result files early |
| The log ends with a long error report right after a score table | The scoring job exits with an error code whenever a target is missed. That is expected | The job scripts ignore it. Read the score table above the error report |
| Hindi or multi-turn scores change between identical runs | Those test sets are small (27 and 20 items), and the AI does not give bit-for-bit identical answers | Treat a change of one or two items as chance |
| "Failed to save draft" at the top of the page | A short-lived save problem in Kaggle's editor | Harmless; it clears itself |
| The page freezes when you paste a very long line | Kaggle's editor struggles with it | Keep cells short; long logic lives in `scripts/` |
| An error mentions `HF_TOKEN` | The secret is not attached to this notebook | Add-ons > Secrets, attach `HF_TOKEN` |
| Jobs stop being able to download the scripts | The GitHub project was public only so Kaggle could read it. If it is made private, the download needs a key | When the project goes private, add a GitHub key to `start_job` |

## 10. Where things are in the project

| Path | What it is |
|---|---|
| `scripts/kaggle_train.py` | The Kaggle job that sets up, rebuilds the data, trains, uploads the model and scores it |
| `scripts/kaggle_eval.py` | The Kaggle job that sets up, rebuilds the notes and scores an existing model |
| `scripts/merge_tejas_data.py` | Builds the training file from Tejas's data, our earlier rows and the computer-made rows; sets aside old-law rows and rows waiting for review |
| `scripts/split_data.py` | Splits the training file 90/10 into training and held-back examples |
| `scripts/generate_calc_rows.py` | Creates the 312 calculation rows |
| `scripts/build_kb.py` | Builds the notes file the lookup searches (`data/knowledge/kb.jsonl`) |
| `training/sft_train.py` | The training program (settings: `SEED`, `MAX_STEPS`, `HF_REPO`) |
| `training/COMPARE_MODELS.md` | How to compare two starting models fairly |
| `eval/run_full_eval.py` | The scoring program: 192 English questions, Hindi set, multi-turn set, wrong-refusal check |
| `eval/run_fact_check.py` | Checks 50 of Tejas's facts. `--notes-only` needs no graphics card |
| `eval/golden_set.jsonl` | The 192 English test questions (GST, Payroll, Invoice, Refusal) |
| `guardrails/retrieval.py` | The lookup (word and meaning search) and the Sources line |
| `guardrails/calculators.py` | The calculators |
| `guardrails/answer_support.py` | Connects the lookup, calculators and Sources line to the live server; settings `KINNOW_LOOKUP`, `KINNOW_CALC`, `KINNOW_SOURCES`, `KINNOW_LOOKUP_K`, `KINNOW_KB_PATH` |
| `guardrails/pipeline.py` | The four safety layers the live server applies to every question |
| `data/tejas/`, `data/tejas_review/` | Tejas's data and checklists, and `DECISIONS.md` (the log of disputed legal points) |
| `data/v2_training/` | The current training file and the lists of held or waiting rows |
| `data/review/` | Kept on one computer only, not in GitHub: the 50-answer review sheet for Tejas, its answer key, the saved v4 answers and the failure sort |
| `kinnow-production-plan-v1.md` | The plan and its targets |

## 11. Glossary in plain words

| Term | Meaning |
|---|---|
| Model | The AI program that writes the answers (here, Qwen2.5-7B) |
| Training / fine-tuning | Teaching the model with our own reviewed questions and answers |
| QLoRA | A light training method that changes only a small part of the model, so it fits on small graphics cards |
| Trained model ("adapter") | The small file of changes that training produces. It is used together with the original model |
| Lookup | Before answering, the system searches our approved notes for the ones that fit the question and shows them to the model |
| Better ("hybrid") lookup | A lookup that searches by exact words and by meaning, then merges the results |
| Sources line | A line such as "Sources: Section 16(2)(aa), Rule 37" added after the answer, from the notes the model was shown |
| Test questions ("golden set") | The 192 fixed English questions every version is scored on. An answer passes if it has the required words and none of the forbidden ones |
| Fact check | 50 facts from Tejas's list, each tested by a question |
| Wrong refusal | The model refuses a real ERP question. Our limit is 3% |
| Held / waiting rows | Training rows waiting for a CA: held rows are left out; "assumed" rows were used on the owner's decision |
| Chance (noise) | Score movement that happens by luck. With about 50 questions per topic, one question is about 2 points |
| Hugging Face | The website where we store trained models. Ours are private |
| Session | One run of the Kaggle notebook on a graphics card. When it ends, its files are deleted |